# Nạp mốc LSTM trên G H I J từ tệp nén trên Drive

Notebook **chỉ đọc Drive**. Không train, không clone mã, không sửa gì trên Drive.

TN4 chấm các cấu hình DS-TCN trên G H I J. Muốn đọc được bảng đó thì phải có mốc
để so, và mốc đó là **LSTM-352** — kiến trúc của bài báo MobiVital. Nó đã train
và chấm xong từ trước, nhưng tệp kết quả chỉ nằm trên Drive.

| mốc | tệp nén cần | chứa |
|---|---|---|
| LSTM-352 | `tn1_lstm.zip` | thư mục `tn1_ghij/` bên trong, 3 seed |

Khác với `NAP_KET_QUA_TN1.ipynb`: notebook kia lấy phần **4 fold CV**, notebook
này lấy phần **test trên G H I J**. Hai phần nằm trong cùng tệp nén nhưng khác
thư mục, khác cách đặt tên — test cuối không có hậu tố fold.

Chạy lần lượt từ trên xuống.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Hai mốc cần lấy

`config_id` dưới đây là tên thật `run_final_test.py` đã sinh ra lúc chạy. Test
cuối train đủ tám người ABCDEFKL rồi chấm 537 phiên của G H I J, nên tên **không
có hậu tố fold** — chỉ có `_seed<N>`.

In [ ]:
SRC = "/content/drive/MyDrive/mobivital"

# config_id (bỏ phần _seed<N>)  ->  tên thư mục trong runs/tn1_ghij/
CAU_HINH = {
    "lstm_mse_corr0.9": "LSTM-352",
}

# Điểm đã biết, dùng để bắt lỗi tệp nén sai hoặc thiếu seed.
MONG_DOI = {"LSTM-352": 0.810302}
NGUONG = 1e-4

N_PHIEN = 537      # số buổi ghi của G H I J. Thiếu là dữ liệu bị cụt.

## 3. Tệp nén trên Drive

In [ ]:
import glob, os

CAN = ["tn1_lstm.zip"]
# Bản dự phòng: tệp nén tích luỹ, bản seed sau chứa cả seed trước.
DU_PHONG = ["tn1_lstm_mse_corr0.9_seed1.zip", "tn1_lstm_mse_corr0.9_seed2.zip"]

co = []
print("CẦN")
for ten in CAN:
    p = os.path.join(SRC, ten)
    if os.path.exists(p):
        co.append(p); print("   %8.1f MB  %s" % (os.path.getsize(p)/1e6, ten))
    else:
        print("   %8s     %s   KHÔNG THẤY" % ("-", ten))

print()
print("DỰ PHÒNG có sẵn")
for ten in DU_PHONG:
    p = os.path.join(SRC, ten)
    if os.path.exists(p):
        co.append(p); print("   %8.1f MB  %s" % (os.path.getsize(p)/1e6, ten))

print()
print("sẽ giải", len(co), "tệp nén")

## 4. Giải nén và xếp lại layout

Tệp nén có dạng phẳng:

```
tn1_ghij/<config_id>/final.pth
tn1_ghij/<config_id>/curve.csv
tn1_ghij/scores_<config_id>.csv
tn1_ghij/<config_id>.txt          <- bảng lựa chọn kênh, 537 dòng
```

Repo dùng dạng lồng theo cấu hình:

```
runs/tn1_ghij/<tên>/seed<N>/{final.pth, curve.csv, scores.csv, selection.txt}
```

Tệp nào đã có thì **so byte**; khác byte là báo xung đột chứ không đè lặng lẽ.

In [ ]:
import hashlib, re, shutil, subprocess, tempfile

DICH = "/content/runs_GHIJ/tn1_ghij"
os.makedirs(DICH, exist_ok=True)

def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()

xung_dot, da_chep = [], 0

def chep(nguon, dich):
    global da_chep
    if os.path.exists(dich):
        if sha(nguon) != sha(dich):
            xung_dot.append(dich)
        return
    os.makedirs(os.path.dirname(dich), exist_ok=True)
    shutil.copy2(nguon, dich)
    da_chep += 1

def tach(ten):
    """'lstm_mse_corr0.9_seed1' -> ('LSTM-352', 1), hoặc None."""
    m = re.match(r"^(.*)_seed(\d+)$", ten)
    if not m or m.group(1) not in CAU_HINH:
        return None
    return CAU_HINH[m.group(1)], int(m.group(2))

tam = tempfile.mkdtemp()
for i, z in enumerate(co):
    subprocess.run(["unzip", "-oq", z, "-d", os.path.join(tam, str(i))], check=True)

for thu_muc, _, ten_tep in os.walk(tam):
    # CHỈ lấy phần tn1_ghij. tn1_lstm.zip nén cả runs/tn1 lẫn runs/tn1_ghij,
    # mà hai bên dùng CÙNG config_id — không lọc thì kết quả 4 fold lẫn vào.
    if "tn1_ghij" not in thu_muc.split(os.sep):
        tren = os.path.basename(os.path.dirname(thu_muc))
        if tren != "tn1_ghij":
            continue

    r = tach(os.path.basename(thu_muc))
    if r:                                   # thư mục <config_id>/
        ten, seed = r
        for t in ten_tep:
            if t in ("final.pth", "curve.csv"):
                chep(os.path.join(thu_muc, t), "%s/%s/seed%d/%s" % (DICH, ten, seed, t))
        continue

    for t in ten_tep:                       # tệp nằm ngang hàng
        if t.startswith("scores_") and t.endswith(".csv"):
            r = tach(t[len("scores_"):-len(".csv")])
            if r:
                chep(os.path.join(thu_muc, t),
                     "%s/%s/seed%d/scores.csv" % (DICH, r[0], r[1]))
        elif t.endswith(".txt") and not t.startswith("README"):
            r = tach(t[:-len(".txt")])
            if r:
                chep(os.path.join(thu_muc, t),
                     "%s/%s/seed%d/selection.txt" % (DICH, r[0], r[1]))

print("chép", da_chep, "tệp")
if xung_dot:
    print()
    print("XUNG ĐỘT — cùng đường dẫn, khác nội dung:")
    for d in xung_dot:
        print("   ", d)
else:
    print("không xung đột")

## 5. Đủ chưa — phải có 3 seed × 4 tệp

In [ ]:
thieu = []
for ten in CAU_HINH.values():
    dong = []
    for seed in (0, 1, 2):
        n = 0
        for t in ("final.pth", "curve.csv", "scores.csv", "selection.txt"):
            p = "%s/%s/seed%d/%s" % (DICH, ten, seed, t)
            if os.path.exists(p):
                n += 1
            else:
                thieu.append("%s/seed%d/%s" % (ten, seed, t))
        dong.append("seed%d: %d/4" % (seed, n))
    print("  %-12s %s" % (ten, "   ".join(dong)))

print()
if thieu:
    print("THIẾU", len(thieu), "tệp:")
    for t in thieu:
        print("   ", t)
else:
    print("ĐỦ — 3 seed × 4 tệp")

## 6. Kiểm điểm và kiểm số phiên

Test cuối phải chấm **đủ 537 buổi ghi** của G H I J. Chấm 500/537 vẫn ra một con
số trông bình thường, nên phải đếm.

Điểm chính là **Pearson macro theo người**: trung bình theo từng người trước, rồi
trung bình bốn người G H I J.

In [ ]:
import csv, statistics, collections

def cham(duong_dan):
    rows = list(csv.DictReader(open(duong_dan)))
    theo_nguoi = collections.defaultdict(list)
    for r in rows:
        theo_nguoi[r["user"]].append(float(r["pearson"]))
    macro = statistics.mean(statistics.mean(v) for v in theo_nguoi.values())
    micro = statistics.mean(float(r["pearson"]) for r in rows)
    return macro, micro, len(rows), sorted(theo_nguoi)

print("%-12s %-7s %10s %10s %8s  %s" % ("mốc", "seed", "macro", "micro", "phiên", "người"))
print("-" * 70)
ket_qua, du_phien = {}, True
for ten in CAU_HINH.values():
    if not os.path.isdir(DICH + "/" + ten):
        print("%-12s KHÔNG CÓ DỮ LIỆU" % ten); continue
    ma, mi = [], []
    for seed in (0, 1, 2):
        a, b, n, ng = cham("%s/%s/seed%d/scores.csv" % (DICH, ten, seed))
        ma.append(a); mi.append(b)
        du_phien &= (n == N_PHIEN)
        print("%-12s %-7d %10.6f %10.6f %8d  %s"
              % (ten, seed, a, b, n, "".join(ng) + ("" if n == N_PHIEN else "  THIẾU PHIÊN")))
    ket_qua[ten] = (statistics.mean(ma), statistics.stdev(ma),
                    statistics.mean(mi), statistics.stdev(mi))
    print("%-12s %-7s %10.6f %10.6f" % ("", "gộp", ket_qua[ten][0], ket_qua[ten][2]))
    print()

print("=" * 70)
print("%-12s %22s %22s" % ("mốc", "macro mean ± std", "micro mean ± std"))
ok = du_phien
for ten, (m, sm, u, su) in ket_qua.items():
    hop = ""
    if ten in MONG_DOI:
        dat = abs(m - MONG_DOI[ten]) < NGUONG
        ok &= dat
        hop = "ĐẠT" if dat else "LỆCH (mong %.6f)" % MONG_DOI[ten]
    print("%-12s   %.6f ± %.6f     %.6f ± %.6f   %s" % (ten, m, sm, u, su, hop))
print()
print("số phiên đủ 537 ở mọi lượt:", du_phien)
print("TẤT CẢ ĐẠT" if ok else "CÓ CHỖ CẦN XEM LẠI")

## 7. Gộp `summary.csv`

Chỉ lấy dòng có `experiment` bằng `tn1_ghij`. Cột `device` xoá trắng — tên đời
GPU không đổi kết quả nào mà lại dính vào mọi bảng.

In [ ]:
COT = ["run_id", "timestamp", "git_commit", "device",
       "experiment", "model", "loss", "alpha",
       "corr_threshold", "seed", "fold", "val_users",
       "n_params", "n_train_windows", "epochs",
       "train_mse", "train_pearson", "train_loss", "minutes_train", "resumed",
       "score_macro", "score_micro", "score_std", "n_sessions", "n_negative",
       "minutes_score"]

GIU = {"%s_seed%d" % (goc, s) for goc in CAU_HINH for s in (0, 1, 2)}

dong, da_thay = [], set()
for s in glob.glob(tam + "/**/summary.csv", recursive=True):
    for r in csv.DictReader(open(s)):
        rid = r.get("run_id", "")
        if r.get("experiment") != "tn1_ghij" or rid not in GIU or rid in da_thay:
            continue
        da_thay.add(rid)
        r["device"] = ""
        dong.append({k: r.get(k, "") for k in COT})

# Dòng thiếu thì dựng lại từ scores.csv — đúng cách run_final_test.py tính.
for goc, ten in CAU_HINH.items():
    for seed in (0, 1, 2):
        rid = "%s_seed%d" % (goc, seed)
        p = "%s/%s/seed%d/scores.csv" % (DICH, ten, seed)
        if rid in da_thay or not os.path.exists(p):
            continue
        macro, micro, n, _ = cham(p)
        moi = {k: "" for k in COT}
        moi.update({"run_id": rid, "experiment": "tn1_ghij", "seed": str(seed),
                    "val_users": "GHIJ", "model": goc.split("_")[0],
                    "loss": "mse", "alpha": "1.0", "corr_threshold": "0.9",
                    "epochs": "20", "n_sessions": str(n),
                    "score_macro": repr(macro), "score_micro": repr(micro)})
        dong.append(moi); da_thay.add(rid)
        print("dựng lại dòng thiếu:", rid, "-> %.6f" % macro)

dong.sort(key=lambda r: (r["run_id"]))
with open("/content/runs_GHIJ/tn1_ghij/summary_bo_sung.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COT)
    w.writeheader(); w.writerows(dong)

print("gom", len(dong), "dòng (cần 3)")
print("còn tên GPU:", sum(1 for r in dong if r["device"]))

## 8. Đóng gói và tải về

In [ ]:
subprocess.run(["bash", "-lc",
                "cd /content && tar -czf runs_GHIJ.tar.gz runs_GHIJ"], check=True)
print("%.1f MB" % (os.path.getsize("/content/runs_GHIJ.tar.gz") / 1e6))
print()
print(subprocess.run(["bash", "-lc",
      "cd /content && find runs_GHIJ | sort"], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import files
files.download("/content/runs_GHIJ.tar.gz")

## 9. Gửi tệp `runs_GHIJ.tar.gz` về

Không cần làm gì thêm ở máy — gửi tệp tải về là đủ.

Ô 6 đã tính lại điểm từ `scores.csv`, đếm đủ 537 phiên, và so LSTM-352 với con số
đã biết. Ô đó in `TẤT CẢ ĐẠT` thì tệp dùng được luôn.